<a href="https://colab.research.google.com/github/venkateshsunkula/pre-recall-radar/blob/main/pre_recall_radar.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pre-Recall Radar
An AI agent that detects emerging product safety issues from public CPSC complaint data, before official recalls.

## Step 1: Download CPSC data
Full public database from SaferProducts.gov (complaints + recalls).

In [35]:
!wget -O SPDB.zip https://www.saferproducts.gov/SPDB.zip
!unzip -o SPDB.zip
!ls

--2026-10-02 02:22:26--  https://www.saferproducts.gov/SPDB.zip
Resolving www.saferproducts.gov (www.saferproducts.gov)... 173.222.226.186, 2600:1406:5e00:389::983, 2600:1406:5e00:399::983
Connecting to www.saferproducts.gov (www.saferproducts.gov)|173.222.226.186|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 39687441 (38M) [application/x-zip-compressed]
Saving to: ‘SPDB.zip’

SPDB.zip            100%[===================>]  37.85M   113MB/s    in 0.3s    

2026-10-02 02:22:27 (113 MB/s) - ‘SPDB.zip’ saved [39687441/39687441]

Archive:  SPDB.zip
  inflating: Disclaimer.txt          
  inflating: IncidentReports.csv     
  inflating: Recalls.csv             
Disclaimer.txt	IncidentReports.csv  Recalls.csv  sample_data  SPDB.zip


## Step 2: Load complaints
The CSV has a disclaimer line above the header, so we find the real header row first.

In [36]:
   import pandas as pd
   with open("IncidentReports.csv", encoding="latin-1") as f:
    for i, line in enumerate(f):
        if "Report No." in line:
            header_row = i
            break

print("Real header is on line", header_row)

df = pd.read_csv("IncidentReports.csv", skiprows=header_row,
                 low_memory=False, encoding="latin-1")
df.columns = df.columns.str.strip()

print(df.shape)
print([c for c in df.columns if "Category" in c])

Real header is on line 1
(70000, 40)
['Category of Submitter', 'Product Category', 'Product Sub Category']


## Step 3: Focus on baby products

In [37]:
print(df["Product Category"].value_counts().head(20))

Product Category
Kitchen                                 25216
Home Maintenance and Structures         11740
Furniture, Furnishings & Decorations     6021
Electronics                              5429
Baby                                     4634
Sports and Recreation                    4482
Toys & Children                          3611
Yard & Garden                            3451
Clothing and Accessories                 1938
Personal Care                            1197
Containers & Packaging                   1129
Fuel, Lighters and Fireworks              575
Hobby                                     427
Products at Public Facilities              95
Drywall                                    55
Name: count, dtype: int64


In [38]:
baby = df[
    df["Product Category"].str.contains("nursery|infant|baby", case=False, na=False)
    | df["Product Sub Category"].str.contains("nursery|infant|baby|crib|stroller|car seat", case=False, na=False)
].copy()

baby["Report Date"] = pd.to_datetime(baby["Report Date"], errors="coerce")

print(len(baby), "baby product complaints")
print(baby["Product Type"].value_counts().head(15))
baby[["Report Date", "Brand", "Product Type", "Incident Description"]].head()

5286 baby product complaints
Product Type
Diapers (1512)                                                         894
Bassinets or Cradles (1537)                                            597
Baby Strollers (1522)                                                  595
Nonportable Cribs (1543)                                               452
Pacifiers or Teething Rings (1525)                                     430
High Chairs (1555)                                                     255
Baby Exercisers (1520)                                                 211
Infant & Toddler Play Ctrs, Excl Jumpers,bouncers&exercisers (1550)    176
Baby Walkers or Jumpers (1508)                                         170
Baby Gates or Barriers (1506)                                          165
Night-lights (1533)                                                    151
Playpens (1513)                                                        149
Baby Bouncer Seats (Excl. Jumpers) (1558)                 

,Report Date,Brand,Product Type,Incident Description
1,2026-09-11,Millie Moon,Diapers (1512),My infant daughter suffered severe burns and s...
2,2026-09-12,millie moon,Diapers (1512),I have been using MM size 4 diapers since my c...
3,2026-09-12,Millie moon,Diapers (1512),I started using Millie moon diapers a few week...
4,2026-09-12,MillieMoon Luxury Daiper,Diapers (1512),When our daughter was two weeks we started usi...
5,2026-09-14,Millie moon,Diapers (1512),Millie moon diapers causing extreme diaper ras...


## Step 4: Spike detector
Flags products with 3+ complaints in the last 90 days AND 3x their usual monthly rate.

In [39]:
# 1. Clean brand names: lowercase, letters only ("Millie Moon" -> "milliemoon")
baby["brand_key"] = (baby["Brand"].astype(str).str.lower()
                     .str.replace(r"[^a-z]", "", regex=True))

# 2. Define time windows
today = pd.Timestamp("2026-05-15")              # newest complaint = "today"
recent_start = today - pd.Timedelta(days=90)     # last 3 months
before_start = recent_start - pd.Timedelta(days=365)  # the year before that

recent = baby[(baby["Report Date"] > recent_start) &
              (baby["Report Date"] <= today)]
before = baby[(baby["Report Date"] > before_start) &
              (baby["Report Date"] <= recent_start)]

# 3. Count complaints per product in each window
keys = ["brand_key", "Product Type"]
spikes = pd.concat([
    recent.groupby(keys).size().rename("recent_90d"),
    before.groupby(keys).size().rename("prior_365d"),
], axis=1).fillna(0)

# 4. Flag: at least 3 recent complaints AND 3x more than usual
spikes["recent_per_month"] = spikes["recent_90d"] / 3
spikes["usual_per_month"] = spikes["prior_365d"] / 12
spikes["flag"] = ((spikes["recent_90d"] >= 3) &
                  (spikes["recent_per_month"] >= 3 * spikes["usual_per_month"]))

flagged = spikes[spikes["flag"]].sort_values("recent_90d", ascending=False)
print(len(flagged), "products flagged")
flagged.head(15)


4 products flagged


,,recent_90d,prior_365d,recent_per_month,usual_per_month,flag
brand_key,Product Type,,,,,
milliemoon,Diapers (1512),35.0,2.0,11.666667,0.166667,True
flyboss,Baby Bouncer Seats (Excl. Jumpers) (1558),3.0,2.0,1.000000,0.166667,True
joolbaby,Potty Chairs or Training Seats (1535),3.0,0.0,1.000000,0.000000,True
milliemoonluxurydiapers,Diapers (1512),3.0,0.0,1.000000,0.000000,True


## Step 5: Claude as the judge
Claude reads complaints and decides if they describe the same hazard. API key is stored in Colab Secrets, never in code.

In [40]:
!pip install -q anthropic
import anthropic, json
from google.colab import userdata
client = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"))

In [41]:
def judge(brand_key, product_type):
    rows = recent[(recent["brand_key"] == brand_key) &
                  (recent["Product Type"] == product_type)]
    texts = rows["Incident Description"].dropna().astype(str).str[:500].head(15).tolist()
    complaints = "\n\n".join(f"- {t}" for t in texts)

    prompt = f"""You are a product-safety analyst. Below are recent consumer complaints about one product.

Brand as typed by consumers: {brand_key}
Product type: {product_type}
Number of complaints: {len(texts)}

Complaints:
{complaints}

Rules:
- canonical_brand must be a cleaned-up version of the brand as typed above, company/brand name ONLY (drop product-line words like "Luxury", "Diapers", "Size 4"). Never use a different brand mentioned in the complaint text.
- If same_hazard is false, severity must be "low".
- If there are fewer than 5 complaints, confidence must be 0.5 or lower.

Reply with ONLY a JSON object, no other text:
{{"canonical_brand": "clean brand name",
 "same_hazard": true or false,
 "hazard": "short description of the main hazard, or null",
 "severity": "low" or "medium" or "high",
 "confidence": number between 0 and 1,
 "summary": "2 factual sentences. Say 'reports describe', not 'the product causes'."}}"""

    resp = client.messages.create(model="claude-haiku-4-5-20251001", max_tokens=500,
                                  messages=[{"role": "user", "content": prompt}])
    text = resp.content[0].text.strip()
    text = text.removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    return json.loads(text)

results = []
for (brand_key, product_type), row in flagged.iterrows():
    verdict = judge(brand_key, product_type)
    verdict.update({"brand_key": brand_key, "recent_90d": int(row["recent_90d"])})
    results.append(verdict)
    print(verdict["canonical_brand"], "|", verdict["hazard"], "|", verdict["severity"])

judged = pd.DataFrame(results)
judged

Millie Moon | Severe skin irritation, chemical burns, rashes, welts, blisters, and open sores in diaper contact areas following a recent product formulation change | high
Flyboss | Baby bouncer seat collapses during use, causing infants to fall and strike their heads and bodies on the floor and frame components | high
Jool Baby | Sharp edges on splash guard causing laceration injuries to genitals when children dismount the potty chair | high
Millie Moon | Chemical burn or severe contact dermatitis from diaper material | high


,canonical_brand,same_hazard,hazard,severity,confidence,summary,brand_key,recent_90d
0,Millie Moon,True,"Severe skin irritation, chemical burns, rashes...",high,0.95,Reports describe 15 complaints of severe skin ...,milliemoon,35
1,Flyboss,True,"Baby bouncer seat collapses during use, causin...",high,0.50,Reports describe a Flyboss baby bouncer seat t...,flyboss,3
2,Jool Baby,True,Sharp edges on splash guard causing laceration...,high,0.50,Reports describe three incidents where toddler...,joolbaby,3
3,Millie Moon,True,Chemical burn or severe contact dermatitis fro...,high,0.50,Reports describe severe chemical burns or cont...,milliemoonluxurydiapers,3


### Evaluation: checking Claude's output
Claude invented a brand name that appears in none of the complaints, so it was a hallucination. Fixed by adding rules to the prompt.

In [42]:
rows = recent[recent["brand_key"] == "millemoon"]
print(rows["Brand"].unique())
for t in rows["Incident Description"].astype(str).head(6):
    print("-", t[:300], "\n")

['Mille Moon']
- My 8 month old son used 20-25 size 3 Millie Moon brand luxury diapers purchased in April 2026. After several days of use he had his first case of diaper rash. It spread up his left buttock leaving red welts of flakey skin that seems painful. We switched diaper brands and while the diaper rash near h 



In [43]:
print(rows["Incident Description"].astype(str).str.contains("mama bear", case=False).sum())

0


## Step 6: Final alerts
Keep only real patterns and merge brand spellings.

In [44]:
alerts = (judged[judged["same_hazard"] == True]
          .groupby("canonical_brand")
          .agg(complaints_90d=("recent_90d", "sum"),
               hazard=("hazard", "first"),
               severity=("severity", "first"),
               summary=("summary", "first"))
          .sort_values("complaints_90d", ascending=False))
alerts

,complaints_90d,hazard,severity,summary
canonical_brand,,,,
Millie Moon,38,"Severe skin irritation, chemical burns, rashes...",high,Reports describe 15 complaints of severe skin ...
Flyboss,3,"Baby bouncer seat collapses during use, causin...",high,Reports describe a Flyboss baby bouncer seat t...
Jool Baby,3,Sharp edges on splash guard causing laceration...,high,Reports describe three incidents where toddler...


## Step 7: Recall check

In [45]:
def recall_status(brand):
    hits = [line.strip()[:200]
            for line in open("Recalls.csv", encoding="latin-1")
            if brand.lower() in line.lower()]
    status = "RECALLED" if hits else "⚠️ EARLY SIGNAL - no recall found"
    return status, hits

for brand in alerts.index:
    status, hits = recall_status(brand)
    alerts.loc[brand, "status"] = status
    print(brand, "→", status)
    for h in hits[:3]:
        print("    ", h)

alerts

Millie Moon → ⚠️ EARLY SIGNAL - no recall found
Flyboss → ⚠️ EARLY SIGNAL - no recall found
Jool Baby → RECALLED
     Jool Baby Recalls Nova Baby Infant Swings Due to Suffocation Hazard; Violation of the Federal Safety Regulations,3/7/2024,"The recall involves Jool Baby's Nova Baby infant swings, manufactured from Ju


,complaints_90d,hazard,severity,summary,status
canonical_brand,,,,,
Millie Moon,38,"Severe skin irritation, chemical burns, rashes...",high,Reports describe 15 complaints of severe skin ...,⚠️ EARLY SIGNAL - no recall found
Flyboss,3,"Baby bouncer seat collapses during use, causin...",high,Reports describe a Flyboss baby bouncer seat t...,⚠️ EARLY SIGNAL - no recall found
Jool Baby,3,Sharp edges on splash guard causing laceration...,high,Reports describe three incidents where toddler...,RECALLED


## Step 8: Backtest
Replays history week by week, using only data that was public at the time.

In [46]:
baby["Publication Date"] = pd.to_datetime(baby["Publication Date"], errors="coerce")

def is_flagged(day, brand="milliemoon", date_col="Publication Date"):
    d = baby[date_col]
    rs = day - pd.Timedelta(days=90)
    bs = rs - pd.Timedelta(days=365)
    mine = baby["brand_key"] == brand
    recent_n = (mine & (d > rs) & (d <= day)).sum()
    before_n = (mine & (d > bs) & (d <= rs)).sum()
    flag = recent_n >= 3 and (recent_n / 3) >= 3 * (before_n / 12)
    return flag, recent_n, before_n

for col in ["Report Date", "Publication Date"]:
    for day in pd.date_range("2026-01-01", "2026-09-30", freq="W"):
        flag, r, b = is_flagged(day, date_col=col)
        if flag:
            print(f"{col}: first flagged {day.date()} | recent 90d: {r} | prior year: {b}")
            break

Report Date: first flagged 2026-03-29 | recent 90d: 3 | prior year: 1
Publication Date: first flagged 2026-04-26 | recent 90d: 3 | prior year: 1


### Lead time vs. false alarms
Watch level (3+) = early but noisy. Alert level (10+) = later but precise.

In [47]:
def first_flag(min_n, brand="milliemoon", col="Publication Date"):
    for day in pd.date_range("2026-01-01", "2026-09-30", freq="W"):
        d = baby[col]
        rs = day - pd.Timedelta(days=90); bs = rs - pd.Timedelta(days=365)
        mine = baby["brand_key"] == brand
        r = (mine & (d > rs) & (d <= day)).sum()
        b = (mine & (d > bs) & (d <= rs)).sum()
        if r >= min_n and r/3 >= 3 * b/12:
            return day.date(), r

def false_alarms(min_n, col="Publication Date"):
    d = baby[col]
    flagged = set()
    for day in pd.date_range("2026-01-01", "2026-09-30", freq="W"):
        rs = day - pd.Timedelta(days=90); bs = rs - pd.Timedelta(days=365)
        r = baby[(d > rs) & (d <= day)].groupby("brand_key").size()
        b = baby[(d > bs) & (d <= rs)].groupby("brand_key").size()
        both = pd.concat([r.rename("r"), b.rename("b")], axis=1).fillna(0)
        hit = both[(both.r >= min_n) & (both.r/3 >= 3 * both.b/12)].index
        flagged.update(hit)
    return len(flagged)

for level, n in [("Watch", 3), ("Alert", 10)]:
    print(f"{level} (>= {n}): Millie Moon first flagged {first_flag(n)} | "
          f"total brands flagged Jan-Sep: {false_alarms(n)}")

Watch (>= 3): Millie Moon first flagged (datetime.date(2026, 4, 26), np.int64(3)) | total brands flagged Jan-Sep: 14
Alert (>= 10): Millie Moon first flagged (datetime.date(2026, 5, 24), np.int64(14)) | total brands flagged Jan-Sep: 3


### Which brands triggered the Alert level?

In [48]:
d = baby["Publication Date"]
alert_brands = set()
for day in pd.date_range("2026-01-01", "2026-09-30", freq="W"):
    rs = day - pd.Timedelta(days=90); bs = rs - pd.Timedelta(days=365)
    r = baby[(d > rs) & (d <= day)].groupby("brand_key").size()
    b = baby[(d > bs) & (d <= rs)].groupby("brand_key").size()
    both = pd.concat([r.rename("r"), b.rename("b")], axis=1).fillna(0)
    alert_brands.update(both[(both.r >= 10) & (both.r/3 >= 3 * both.b/12)].index)
print(alert_brands)


{'ciuseianx', 'milliemoonluxurydiapers', 'milliemoon'}


### Investigating the second alert
Infant walkers: CPSC issued a safety warning on Aug 6, 2026. The agent alerted on Jul 5, 32 days earlier.

In [49]:
c = baby[baby["brand_key"] == "ciuseianx"]
print(len(c), "complaints |", c["Product Type"].unique())
for t in c["Incident Description"].astype(str).head(5):
    print("-", t[:250], "\n")
print(recall_status("ciuseianx"))

11 complaints | ['Baby Exercisers (1520)' 'Baby Walkers or Jumpers (1508)']
- We are a distributor of maternity and baby products operating on Amazon US. We only engage in product distribution and are not a manufacturer. Back in [REDACTED], we sold infant walkers covered under Recall [REDACTED]. From [REDACTED] to the present, 

- I bought a baby walker for my sisters child on Amazon. Id come across some nice baby walkers on [REDACTED] before, and my nephew was old enough to use one. So I placed an order on Amazon. But it ended up being a terrible mistake. What happened is som 

- I bought this product last month, planning to take my daughter on vacation with it this month. Id hoped to watch her use the walker while we were away. I was initially drawn to its appearance, but what happened over the past month is something I'll n 

- My grandson is turning six months old soon. I picked out a lovely gift for his baby shower and specially bought a baby walker for him. I hoped it would help 

In [50]:
print("Watch:", first_flag(3, brand="ciuseianx"))
print("Alert:", first_flag(10, brand="ciuseianx"))

Watch: (datetime.date(2026, 6, 7), np.int64(4))
Alert: (datetime.date(2026, 7, 5), np.int64(10))
